# COX-1 vs COX-2: Can Docking Explain Selectivity?
Aspirin and most painkillers block both cyclooxygenase enzymes. COX-1 protects the stomach lining; COX-2 drives inflammation. "COX-2 selective" drugs like celecoxib were designed to hit COX-2 more than COX-1 to spare the stomach.

The key difference is a **side pocket** in COX-2: position 523 is a small valine (Val523) instead of the bulkier isoleucine in COX-1 (Ile523), and position 513 is an arginine instead of a histidine. Selective drugs fit into that extra space.

**Plan**
1. Prepare both enzymes (COX-1: 1EQG, COX-2: 3LN1 with celecoxib)
2. Validate by re-docking each enzyme's own ligand
3. Dock 7 remedies into both enzymes
4. Compare scores, side-pocket contacts, and known selectivity
5. Be honest about what docking can and cannot tell us

## 1. Setup

In [5]:
!apt-get -qq install -y openbabel > /dev/null
!pip install -q vina rdkit py3Dmol pandas matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for vina (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for vina
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> vina


In [6]:
import os, subprocess, requests
import numpy as np, pandas as pd, matplotlib.pyplot as plt, py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
from vina import Vina

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[:500])
    return r

def xyz(ls):
    return np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in ls])

def read_xyz_pdb(path):
    return xyz([l for l in open(path) if l.startswith(("ATOM", "HETATM"))])

ModuleNotFoundError: No module named 'py3Dmol'

## 2. Load both enzymes
The binding site of each is defined by the inhibitor in its crystal structure. Water, heme, and sugars are removed.

In [ ]:
SKIP = {"HOH", "HEM", "NAG", "BMA", "MAN", "FUC", "GOL", "SO4", "EDO", "BOG", "NA", "CA", "CL"}

def load_receptor(key, pdb_id, prefer):
    text = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30).text
    lines = text.splitlines()
    groups = {}
    for l in lines:
        if l.startswith("HETATM") and l[16] in " A" and l[17:20].strip() not in SKIP:
            groups.setdefault((l[17:20].strip(), l[21]), []).append(l)
    choice = next((k for p in prefer for k in groups if k[0] == p), None)
    if choice is None:
        choice = max(groups, key=lambda k: len(groups[k]))
    resname, chain = choice
    lig = groups[choice]
    protein = [l for l in lines if l.startswith("ATOM") and l[21] == chain and l[16] in " A"]
    open(f"receptor_{key}.pdb", "w").write("\n".join(protein) + "\nEND\n")
    sh(f"obabel receptor_{key}.pdb -O receptor_{key}.pdbqt -xr -h --partialcharge gasteiger")
    return {"key": key, "pdb_id": pdb_id, "resname": resname, "lig": lig, "protein": protein,
            "center": xyz(lig).mean(axis=0), "pxyz": xyz(protein),
            "labels": [(l[17:20].strip(), int(l[22:26])) for l in protein]}

RECS = {
    "cox1": load_receptor("cox1", "1EQG", ["IBP"]),
    "cox2": load_receptor("cox2", "3LN1", ["CEL", "S58"]),
}
for k, r in RECS.items():
    print(k, r["pdb_id"], "| site ligand:", r["resname"], "| protein atoms:", len(r["protein"]))

## 3. Docking helpers

In [ ]:
BOX = [24, 24, 24]

def prep_ligand(smiles, name):
    if not os.path.exists(f"{name}.pdbqt"):
        mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
        AllChem.EmbedMolecule(mol, randomSeed=7)
        AllChem.MMFFOptimizeMolecule(mol)
        Chem.MolToMolFile(mol, f"{name}.mol")
        sh(f"obabel {name}.mol -O {name}.pdbqt --partialcharge gasteiger")
    return f"{name}.pdbqt"

def dock(smiles, name, rec, exhaustiveness=12):
    lig = prep_ligand(smiles, name)
    v = Vina(sf_name="vina", seed=42)
    v.set_receptor(f"receptor_{rec['key']}.pdbqt")
    v.set_ligand_from_file(lig)
    v.compute_vina_maps(center=rec["center"].tolist(), box_size=BOX)
    v.dock(exhaustiveness=exhaustiveness, n_poses=9)
    out = f"{name}_{rec['key']}"
    v.write_poses(f"{out}_out.pdbqt", n_poses=9, overwrite=True)
    sh(f"obabel {out}_out.pdbqt -O {out}_best.pdb -f 1 -l 1")
    return v.energies()[0][0], f"{out}_best.pdb"

def contacts(rec, pose_file, cutoff=4.0):
    lxyz = read_xyz_pdb(pose_file)
    d = np.linalg.norm(rec["pxyz"][:, None, :] - lxyz[None, :, :], axis=2).min(axis=1)
    return {rec["labels"][i] for i in np.where(d < cutoff)[0]}

## 4. Validation: re-dock each enzyme's own inhibitor
A trustworthy setup should reproduce the crystal pose (RMSD under about 2 Å).

In [ ]:
TEMPL = {
    "IBP": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "CEL": "Cc1ccc(cc1)c1cc(nn1c1ccc(cc1)S(N)(=O)=O)C(F)(F)F",
    "S58": "NS(=O)(=O)c1ccc(cc1)n1nc(cc1c1ccc(Br)cc1)C(F)(F)F",
}

def validate(rec):
    smi = TEMPL.get(rec["resname"])
    if smi is None:
        print(rec["key"], ": no template for", rec["resname"], "- skipping RMSD")
        return None
    score, pose_file = dock(smi, f"ref_{rec['key']}", rec)
    t = Chem.MolFromSmiles(smi)
    crystal = AllChem.AssignBondOrdersFromTemplate(t, Chem.MolFromPDBBlock("\n".join(rec["lig"]), removeHs=True))
    pose = AllChem.AssignBondOrdersFromTemplate(t, Chem.MolFromPDBBlock(open(pose_file).read(), removeHs=True))
    try:
        rmsd = rdMolAlign.CalcRMS(pose, crystal)
    except Exception as e:
        rmsd = np.nan; print("RMSD failed:", e)
    print(f"{rec['key']} ({rec['resname']}): score {score:.2f} kcal/mol | RMSD vs crystal {rmsd:.2f} Å")
    return rmsd

val = {k: validate(r) for k, r in RECS.items()}

## 5. Dock seven remedies into both enzymes
This runs 14 dockings and takes roughly 10-20 minutes on Colab's CPU.

In [ ]:
drugs = {
    "Aspirin":     "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen":   "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Naproxen":    "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
    "Diclofenac":  "OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl",
    "Indomethacin": "COc1ccc2n(C(=O)c3ccc(Cl)cc3)c(C)c(CC(=O)O)c2c1",
    "Celecoxib":   "Cc1ccc(cc1)c1cc(nn1c1ccc(cc1)S(N)(=O)=O)C(F)(F)F",
    "Rofecoxib":   "CS(=O)(=O)c1ccc(cc1)C1=C(C(=O)OC1)c1ccccc1",
}
# Approximate classes from the pharmacology literature
known = {"Aspirin": "COX-1 preferring", "Ibuprofen": "non-selective", "Naproxen": "non-selective",
         "Diclofenac": "slightly COX-2 preferring", "Indomethacin": "COX-1 preferring",
         "Celecoxib": "COX-2 selective", "Rofecoxib": "COX-2 selective"}

rows, poses = [], {}
for name, smi in drugs.items():
    row = {"Drug": name}
    for key, rec in RECS.items():
        score, pf = dock(smi, name.lower(), rec)
        poses[(name, key)] = pf
        near = contacts(rec, pf)
        row[f"{key} score"] = round(score, 2)
        row[f"{key} side pocket"] = any(i in (513, 523) for _, i in near)
    row["Δ (COX-2 − COX-1)"] = round(row["cox2 score"] - row["cox1 score"], 2)
    row["Known class"] = known[name]
    rows.append(row)
df = pd.DataFrame(rows).sort_values("Δ (COX-2 − COX-1)")
df.to_csv("selectivity_results.csv", index=False)
df

**Reading the table:** scores are in kcal/mol (more negative = stronger predicted binding). A negative Δ means the drug scores better in COX-2. "Side pocket" shows whether the drug has atoms within 4 Å of residue 513 or 523.

## 6. Plots

In [ ]:
x = np.arange(len(df)); w = 0.38
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(x - w/2, df["cox1 score"], w, label="COX-1", color="#4C78A8")
ax[0].bar(x + w/2, df["cox2 score"], w, label="COX-2", color="#E45756")
ax[0].set_xticks(x); ax[0].set_xticklabels(df["Drug"], rotation=45, ha="right")
ax[0].set_ylabel("Docking score (kcal/mol)"); ax[0].legend(); ax[0].set_title("Binding scores")
colors = ["#E45756" if d < 0 else "#4C78A8" for d in df["Δ (COX-2 − COX-1)"]]
ax[1].bar(x, df["Δ (COX-2 − COX-1)"], color=colors)
ax[1].axhline(0, color="black", lw=0.8)
ax[1].set_xticks(x); ax[1].set_xticklabels(df["Drug"], rotation=45, ha="right")
ax[1].set_title("Δ score (negative = favors COX-2)")
plt.tight_layout(); plt.savefig("selectivity_plot.png", dpi=150); plt.show()

sel = df[df["Known class"] == "COX-2 selective"]["Δ (COX-2 − COX-1)"].mean()
oth = df[df["Known class"] != "COX-2 selective"]["Δ (COX-2 − COX-1)"].mean()
print(f"Mean Δ, known COX-2-selective drugs: {sel:.2f} kcal/mol")
print(f"Mean Δ, all other drugs:            {oth:.2f} kcal/mol")

**What to look for**
- If selective drugs (celecoxib, rofecoxib) show a more negative Δ and reach the side pocket in COX-2 but not COX-1, docking reproduces the expected trend.
- If the ranking is mixed, that is a real and common result: docking scores have errors of 1-2 kcal/mol, which is as large as many selectivity differences. Report it honestly.

## 7. 3D view: the COX-2 side pocket
Gray: COX-2 · orange sticks: residues 513 and 523 · green: docked celecoxib · yellow: docked aspirin · magenta: crystal inhibitor.

In [ ]:
r2 = RECS["cox2"]
view = py3Dmol.view(width=600, height=500)
view.addModel("\n".join(r2["protein"]), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgray", "opacity": 0.6}})
view.addStyle({"model": 0, "resi": [513, 523]}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.2}})
view.addModel(open(poses[("Celecoxib", "cox2")]).read(), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
view.addModel(open(poses[("Aspirin", "cox2")]).read(), "pdb")
view.setStyle({"model": 2}, {"stick": {"colorscheme": "yellowCarbon"}})
view.addModel("\n".join(r2["lig"]), "pdb")
view.setStyle({"model": 3}, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.12}})
view.zoomTo({"model": 1})
view.show()

## Limitations
- Rigid receptors, no heme, no water.
- Scores are approximate; selectivity differences are small and often within error.
- COX-1 is the ovine enzyme and COX-2 the mouse enzyme, both standard crystal models; human enzymes differ slightly.
- Some drugs (aspirin, indomethacin) bind with time-dependent or covalent behavior that docking does not capture.
- Known classes in the table are approximate summaries, not exact measurements.

## Next steps
- Add MM-GBSA rescoring or short MD for each complex
- Test more compounds with known IC50 values and compute a rank correlation
- Wrap the workflow in a web app